# Data Acquisition
**COSC 301 Project — Malaysia State-Level Socioeconomic & Health Outcomes**

This notebook downloads raw data from three sources:
1. **OpenDOSM** : household income, poverty, and population by state
2. **MoH Malaysia GitHub** : public health facilities and hospital beds
3. **World Bank API** : national-level health indicators (backup / cross-validation)

All files are saved to `data/raw/`.

In [ ]:
import requests
import pandas as pd
import os

RAW_DIR = "data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

## Source 1: OpenDOSM (`api.data.gov.my`)

**License:** Creative Commons Attribution (CC-BY)  
**Docs:** https://open.dosm.gov.my/api-docs

| Dataset | Description |
|---|---|
| `hh_income_parlimen` | Household income by parliament constituency |
| `hh_poverty_parlimen` | Household poverty by parliament constituency |
| `population_state` | State-level population totals (overall age/sex/ethnicity only) for per-capita normalisation |

In [ ]:
# OpenDOSM: Household Income by Parliament Constituency
BASE_DOSM = "https://api.data.gov.my/data-catalogue"
params = {"id": "hh_income_parlimen", "limit": 2000}
df = pd.DataFrame(requests.get(BASE_DOSM, params=params).json())
df.to_csv(os.path.join(RAW_DIR, "hh_income_parlimen.csv"), index=False)
print(f"✓ hh_income_parlimen: {len(df)} rows")

In [ ]:
# OpenDOSM: Household Poverty by Parliament Constituency
params = {"id": "hh_poverty_parlimen", "limit": 2000}
df = pd.DataFrame(requests.get(BASE_DOSM, params=params).json())
df.to_csv(os.path.join(RAW_DIR, "hh_poverty_parlimen.csv"), index=False)
print(f"✓ hh_poverty_parlimen: {len(df)} rows")

In [ ]:
# OpenDOSM: Population by State
params = {"id": "population_state", "limit": 10000, 
          "filter": "age@overall_age,sex@overall_sex,ethnicity@overall_ethnicity"}
df = pd.DataFrame(requests.get(BASE_DOSM, params=params).json())
df.to_csv(os.path.join(RAW_DIR, "population_state.csv"), index=False)
print(f"✓ population_state: {len(df)} rows")

In [ ]:
# OpenDOSM: Population by State
URL_DATA = 'https://storage.dosm.gov.my/population/population_state.parquet'
df = pd.read_parquet(URL_DATA)
df.to_csv(os.path.join(RAW_DIR, "population_state.csv"), index=False)
print(f"✓ population_state: {len(df)} rows")

## Source 2: MoH Malaysia GitHub

**License:** Malaysian Open Data License (free for research)  
**Docs:** https://github.com/MoH-Malaysia

MoH publishes data directly on GitHub as CSVs. Pulling from the official repo is more reliable than scraping the portal.

| File | Description |
|---|---|
| `moh_facilities.csv` | Public hospitals and health clinics by state |
| `moh_beds.csv` | Hospital bed counts and utilisation by facility |

In [ ]:
# MoH Malaysia: Public Facilities
MOH_BASE = "https://raw.githubusercontent.com/MoH-Malaysia/data-resources-public/main"
url = f"{MOH_BASE}/facilities_master.csv"
df = pd.read_csv(url)
df.to_csv(os.path.join(RAW_DIR, "moh_facilities.csv"), index=False)
print(f"✓ moh_facilities: {len(df)} rows")

# MoH Malaysia: Hospital Beds
url = f"{MOH_BASE}/bedutil_facility.csv"
df = pd.read_csv(url)
df.to_csv(os.path.join(RAW_DIR, "moh_beds.csv"), index=False)
print(f"✓ moh_beds: {len(df)} rows")

## Source 3: World Bank API (National-Level Backup)

**License:** CC-BY 4.0  
**Coverage:** Malaysia (`MYS`), most recent 20 years

National-level indicators for cross-validation and backup.

| Indicator | Column | Description |
|---|---|---|
| `SP.DYN.LE00.IN` | `life_expectancy` | Life expectancy at birth |
| `SP.DYN.IMRT.IN` | `infant_mortality` | Infant mortality (per 1,000 live births) |
| `SH.XPD.CHEX.GD.ZS` | `health_exp_gdp` | Health expenditure (% of GDP) |
| `SH.MED.BEDS.ZS` | `hospital_beds` | Hospital beds (per 1,000 people) |

In [ ]:
WB_BASE = "https://api.worldbank.org/v2/country/MYS/indicator"

# Life Expectancy
data = requests.get(f"{WB_BASE}/SP.DYN.LE00.IN", 
                    params={"format": "json", "per_page": 100, "mrv": 20}).json()
le_rows = [{"year": int(r["date"]), "life_expectancy": r["value"]} 
           for r in (data[1] or []) if r["value"] is not None]
le_df = pd.DataFrame(le_rows).set_index("year")

# Infant Mortality
data = requests.get(f"{WB_BASE}/SP.DYN.IMRT.IN", 
                    params={"format": "json", "per_page": 100, "mrv": 20}).json()
im_rows = [{"year": int(r["date"]), "infant_mortality": r["value"]} 
           for r in (data[1] or []) if r["value"] is not None]
im_df = pd.DataFrame(im_rows).set_index("year")

# Health Expenditure (% of GDP)
data = requests.get(f"{WB_BASE}/SH.XPD.CHEX.GD.ZS", 
                    params={"format": "json", "per_page": 100, "mrv": 20}).json()
hexp_rows = [{"year": int(r["date"]), "health_exp_gdp": r["value"]} 
             for r in (data[1] or []) if r["value"] is not None]
hexp_df = pd.DataFrame(hexp_rows).set_index("year")

# Hospital Beds (per 1,000 people)
data = requests.get(f"{WB_BASE}/SH.MED.BEDS.ZS", 
                    params={"format": "json", "per_page": 100, "mrv": 20}).json()
beds_rows = [{"year": int(r["date"]), "hospital_beds": r["value"]} 
             for r in (data[1] or []) if r["value"] is not None]
beds_df = pd.DataFrame(beds_rows).set_index("year")

# Combine and save
wb_df = pd.concat([le_df, im_df, hexp_df, beds_df], axis=1).reset_index().sort_values("year")
wb_df.to_csv(os.path.join(RAW_DIR, "worldbank_malaysia.csv"), index=False)
print(f"✓ worldbank_malaysia: {len(wb_df)} rows")

In [ ]:
# Quick check: all files saved and have data
import glob
files = glob.glob(os.path.join(RAW_DIR, "*.csv"))
print(f"\n✓ Downloaded {len(files)} CSV files:")
for f in sorted(files):
    size = len(pd.read_csv(f))
    print(f"  - {os.path.basename(f)}: {size} rows")